# Sommelier Colab

Runs branch `colab` with input `samples/sample.wav` and writes results to Google Drive.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/VDT-TurnTaking/sommelier')
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'colab'
WORK_ROOT = Path('/content/sommelier-colab-work')
REPO_DIR = WORK_ROOT / 'podcast-pipeline-refactor'
VENV_DIR = Path('/content/sommelier-venv')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Output root:', OUTPUT_ROOT)
print('Work root:', WORK_ROOT)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/ngocbao220/sommelier.git'
BRANCH = 'colab'

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
!git branch --show-current
!ls -lh samples/sample.wav

In [ ]:
import os

os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
os.environ['VENV_DIR'] = str(VENV_DIR)

!apt-get update -qq
!apt-get install -y -qq ffmpeg python3-venv
!python -m venv "$VENV_DIR"
!"$VENV_DIR/bin/python" -m pip install -q --upgrade pip setuptools wheel
!"$VENV_DIR/bin/python" -m pip install -q --index-url https://download.pytorch.org/whl/cpu torch
!"$VENV_DIR/bin/python" -m pip install -q -r requirements-colab.txt
!"$VENV_DIR/bin/python" - <<'PY'
import torch
print('torch', torch.__version__)
print('torch file', torch.__file__)
PY

In [ ]:
!PYTHON_BIN="$VENV_DIR/bin/python" bash run_pipeline.sh --config config.colab.json --input samples/sample.wav --output "$OUTPUT_ROOT"

In [ ]:
print('Results:', OUTPUT_ROOT)
!find "$OUTPUT_ROOT" -maxdepth 4 -type f | sort | tail -100